# Session 5: Multi-Tool Orchestration

One Foundry agent, three tools: **Code Interpreter**, **File Search**, and **Function Calling**.

Goal: see how the agent picks a tool per request, and what changes in the SDK when multiple tools are
attached at once instead of just one .

In [1]:
import os

PROJECT_ENDPOINT = os.environ.get("PROJECT_ENDPOINT", "https://multiagentjp.services.ai.azure.com/api/projects/proj-default")
MODEL_DEPLOYMENT_NAME = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5")

print("Project endpoint:", PROJECT_ENDPOINT)
print("Model deployment:", MODEL_DEPLOYMENT_NAME)

Project endpoint: https://multiagentjp.services.ai.azure.com/api/projects/proj-default
Model deployment: gpt-5


## 1. Create the client

`AgentsClient` is the entry point for creating agents, threads, messages, and runs. Auth is via
`DefaultAzureCredential`, which picks up your `az login` session.

In [9]:
!pip install azure-cli

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 78.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.3/246.3 kB 16.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
#!pip install azure-identity azure-ai-agents
from azure.identity import DefaultAzureCredential
from azure.ai.agents import AgentsClient

agents_client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
print("Client ready.")

Client ready.


In [ ]:
!az login

## 2. Tool 1 — Function Calling

A plain Python function. `FunctionTool` inspects it and builds the schema the model uses to decide
when to call it. `enable_auto_function_calls` means the SDK executes the function for you when the
model requests it — you don't have to manually intercept the run and call it yourself.

In [4]:
import json

def get_shipping_estimate(destination_country: str, weight_kg: float) -> str:
    """Return a mock shipping cost and delivery estimate for a destination and package weight.

    :param destination_country: Destination country name.
    :param weight_kg: Package weight in kilograms.
    :return: JSON string with cost and estimated delivery days.
    """
    # Mock logic — replace with a real rates API in a production agent.
    base_cost = 12.5 + (weight_kg * 3.2)
    days = 3 if destination_country.lower() in ("usa", "united states", "canada") else 7
    return json.dumps({
        "destination": destination_country,
        "weight_kg": weight_kg,
        "estimated_cost_usd": round(base_cost, 2),
        "estimated_days": days,
    })

user_functions = {get_shipping_estimate}
print("Function tool defined:", get_shipping_estimate.__name__)

Function tool defined: get_shipping_estimate


## 3. Tool 2 — Code Interpreter

Runs Python in a sandbox. We generate a small synthetic CSV so the agent has something to analyze
and chart without needing a real dataset.

In [5]:
import csv

csv_path = "quarterly_sales.csv"
rows = [
    ("Q1", 128000), ("Q2", 141500), ("Q3", 133200), ("Q4", 158900),
]
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["quarter", "revenue_usd"])
    writer.writerows(rows)
print(f"Wrote {csv_path}")

Wrote quarterly_sales.csv


In [6]:
uploaded_csv = agents_client.files.upload_and_poll(file_path=csv_path, purpose="assistants")
print("Uploaded file id:", uploaded_csv.id)

Uploaded file id: assistant-HZcEdUcjHgv94XwqXB3mnG


## 4. Tool 3 — File Search

File Search needs a **vector store**. We create a small text file, upload it, and build a vector
store from it. The agent will retrieve from this when asked a question the text can answer.

In [7]:
policy_text = """Return Policy

Items may be returned within 30 days of delivery for a full refund.
Items must be unused and in original packaging.
Return shipping is free for defective items; otherwise the customer covers return shipping.
Refunds are processed within 5-7 business days after the item is received.
"""
policy_path = "return_policy.md"
with open(policy_path, "w") as f:
    f.write(policy_text)

uploaded_policy = agents_client.files.upload_and_poll(file_path=policy_path, purpose="assistants")
print("Uploaded file id:", uploaded_policy.id)

Uploaded file id: assistant-9c9zxqNtZ8T5wFiACYicFA


In [8]:
vector_store = agents_client.vector_stores.create_and_poll(
    file_ids=[uploaded_policy.id],
    name="session5-policy-store",
)
print("Vector store id:", vector_store.id)

Vector store id: vs_ySoyp8p1CzcFGkr5hXoCSUgO


## 5. Combine all three into one `ToolSet`

This is the actual multi-tool orchestration step. Each tool is added to the same `ToolSet`, then the
whole set is passed to `create_agent` in one call. Compare this to a single-tool agent (Session 4):
there, `tools=` took one tool's `.definitions` directly. With more than one tool, `ToolSet` is what
lets you combine them cleanly, and `enable_auto_function_calls` is required for the function tool to
actually execute during a run rather than just being offered to the model.

In [9]:
from azure.ai.agents.models import CodeInterpreterTool, FileSearchTool, FunctionTool, ToolSet

function_tool = FunctionTool(user_functions)
code_interpreter_tool = CodeInterpreterTool(file_ids=[uploaded_csv.id])
file_search_tool = FileSearchTool(vector_store_ids=[vector_store.id])

toolset = ToolSet()
toolset.add(function_tool)
toolset.add(code_interpreter_tool)
toolset.add(file_search_tool)

agents_client.enable_auto_function_calls(toolset)

agent = agents_client.create_agent(
    model=MODEL_DEPLOYMENT_NAME,
    name="session5-multi-tool-agent",
    instructions=(
        "You are a customer support agent for an online retailer. "
        "Use file search to answer questions about the return policy. "
        "Use the shipping estimate function when asked about shipping cost or delivery time. "
        "Use code interpreter to analyze the uploaded sales CSV or generate charts from it."
    ),
    toolset=toolset,
)
print("Agent created:", agent.id)

Agent created: asst_HD6KDS2VJZixCB0wdh5S7o7V


## 6. Test tool routing

Three different questions, each aimed at a different tool. We create one thread per question so the
runs are easy to inspect independently.

In [ ]:
test_questions = [
    ("file_search", "How many days do I have to return an item, and who pays for return shipping?"),
    ("function_call", "What would it cost to ship a 4.5 kg package to Canada, and how long would it take?"),
    ("code_interpreter", "Using the uploaded sales CSV, which quarter had the highest revenue, and by how much did it beat the lowest quarter?"),
]

results = []
for expected_tool, question in test_questions:
    thread = agents_client.threads.create()
    agents_client.messages.create(thread_id=thread.id, role="user", content=question)
    run = agents_client.runs.create_and_process(thread_id=thread.id, agent_id=agent.id)
    results.append((expected_tool, question, thread.id, run))
    print(f"[{expected_tool}] run status: {run.status}")

## 7. Read back the answers

In [11]:
for expected_tool, question, thread_id, run in results:
    print(f"\n=== Expected tool: {expected_tool} ===")
    print("Q:", question)
    messages = agents_client.messages.list(thread_id=thread_id)
    for m in messages:
        if m.role == "assistant" and m.text_messages:
            print("A:", m.text_messages[-1].text.value)
            break


=== Expected tool: file_search ===
Q: How many days do I have to return an item, and who pays for return shipping?
A: - Return window: 30 days from delivery.  
- Return shipping: free if the item is defective; otherwise, the customer pays.

These details come from our Return Policy 【4:2†return_policy.md】.

=== Expected tool: function_call ===
Q: What would it cost to ship a 4.5 kg package to Canada, and how long would it take?
A: Here’s an estimate for your shipment:
- Destination: Canada
- Package weight: 4.5 kg
- Estimated shipping cost: $26.90 USD
- Estimated delivery time: about 3 business days

Note: This is an estimate. Final rates and delivery windows may vary based on your exact address, carrier selection, and any surcharges (e.g., remote area, fuel).


## 8. Confirm which tool actually fired

The answer text alone doesn't prove which tool ran. Run steps show the actual tool calls per run —
this is the real validation that multi-tool routing worked, not just that the model produced a
plausible-sounding answer.

In [12]:
for expected_tool, question, thread_id, run in results:
    print(f"\n=== Expected tool: {expected_tool} ===")
    steps = agents_client.run_steps.list(thread_id=thread_id, run_id=run.id)
    for step in steps:
        step_details = step.step_details
        step_type = getattr(step_details, "type", None)
        print("  step type:", step_type)
        tool_calls = getattr(step_details, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                print("    tool call type:", tc.type)


=== Expected tool: file_search ===
  step type: message_creation
  step type: tool_calls
    tool call type: file_search

=== Expected tool: function_call ===
  step type: message_creation
  step type: tool_calls
    tool call type: function


If a run step's tool call type doesn't match the expected tool, that's a real finding worth noting in
`session-notes.md` — model routing between tools isn't always deterministic, and it's worth documenting
cases where it picked something unexpected.

## 9. Cleanup

See `cleanup.md` for the full checklist and cost rationale. Minimal version below.

In [13]:
agents_client.delete_agent(agent.id)
agents_client.vector_stores.delete(vector_store.id)
agents_client.files.delete(uploaded_csv.id)
agents_client.files.delete(uploaded_policy.id)
print("Cleanup complete.")

Cleanup complete.
